# SEQ-02 LSTM Language Model

- Module: 09 Sequence Models
- Topic: character-level language modeling with an LSTM, teacher forcing, and beam-search decoding
- Estimated runtime: 8 to 12 minutes on CPU once `torch` and `matplotlib` are installed
- Expected outputs: a compact language model, training-loss snapshots, and generated continuations from greedy and beam-search decoding


## Why switch from a vanilla RNN to an LSTM?

An LSTM augments the hidden state with a memory cell and gates:

$$
f_t = \sigma(\cdot),
\qquad
i_t = \sigma(\cdot),
\qquad
o_t = \sigma(\cdot),
\qquad
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t.
$$

That additive memory path is the core reason LSTMs preserve information better than plain RNNs on long sequences.


In [ ]:
import random

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 11
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## Tiny character corpus and teacher forcing setup

We use a deliberately small in-notebook corpus so the example stays self-contained. During training, the model sees the true prefix and predicts the next character at every position. That is teacher forcing:

$$
\mathcal{L} = -\sum_{t=1}^{T-1} \log p_\theta(x_{t+1} \mid x_{1:t}).
$$

So the input sequence is `x_1, ..., x_{T-1}` and the target sequence is `x_2, ..., x_T`.


In [ ]:
corpus = """
sequence models read symbols one step at a time.
lstms keep a gated memory cell so useful information can survive longer.
teacher forcing trains on the true prefix while decoding predicts the future.
beam search keeps several likely continuations instead of only one.
""".strip().lower()

corpus = (corpus + "\n") * 40
vocab = sorted(set(corpus))
stoi = {ch: idx for idx, ch in enumerate(vocab)}
itos = {idx: ch for ch, idx in stoi.items()}
encoded = torch.tensor([stoi[ch] for ch in corpus], dtype=torch.long)

vocab_size = len(vocab)
print("corpus length:", len(encoded))
print("vocab size:", vocab_size)
print("vocab:", ''.join(vocab))


In [ ]:
sequence_length = 48
batch_size = 16


def decode(indices):
    return ''.join(itos[int(i)] for i in indices)


def make_batch(data: torch.Tensor, batch_size: int, sequence_length: int):
    starts = torch.randint(0, len(data) - sequence_length - 1, (batch_size,))
    x = torch.stack([data[s : s + sequence_length] for s in starts])
    y = torch.stack([data[s + 1 : s + sequence_length + 1] for s in starts])
    return x.to(device), y.to(device)

x_batch, y_batch = make_batch(encoded, batch_size=batch_size, sequence_length=sequence_length)
print("input batch shape:", tuple(x_batch.shape))
print("target batch shape:", tuple(y_batch.shape))
print("teacher-forced input snippet :", repr(decode(x_batch[0][:30])))
print("next-token target snippet   :", repr(decode(y_batch[0][:30])))


## LSTM language model

We use an embedding layer, one LSTM, and a linear decoder head. The model outputs logits for the next character at every timestep.


In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 24, hidden_size: int = 64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, vocab_size)

    def forward(self, x: torch.Tensor, hidden=None):
        emb = self.embedding(x)
        output, hidden = self.lstm(emb, hidden)
        logits = self.head(output)
        return logits, hidden

    def step(self, token: torch.Tensor, hidden=None):
        emb = self.embedding(token.unsqueeze(1))
        output, hidden = self.lstm(emb, hidden)
        logits = self.head(output[:, -1])
        return logits, hidden


model = CharLSTM(vocab_size=vocab_size).to(device)
logits, hidden = model(x_batch)
print("logits shape:", tuple(logits.shape))


## Train with teacher forcing

At training time we never ask the model to feed back its own mistakes. We always provide the true previous character. That keeps optimization stable and makes the learning target explicit, even though generation at inference time will be autoregressive.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=4e-3)
loss_history = []

for step in range(1, 221):
    xb, yb = make_batch(encoded, batch_size=batch_size, sequence_length=sequence_length)
    logits, _ = model(xb)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    loss_history.append(float(loss.item()))
    if step % 55 == 0:
        print(f"step {step:3d} | cross-entropy={loss.item():.4f}")

plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.xlabel("training step")
plt.ylabel("cross-entropy")
plt.title("LSTM language-model training under teacher forcing")
plt.tight_layout()
plt.show()


## Greedy decoding and beam search

At inference time we no longer have the true next token. We must feed back the model's own prediction.

- Greedy decoding keeps only the most likely next token at each step.
- Beam search keeps the top-$k$ partial sequences by cumulative log-probability, which is a cheap way to look beyond the single best local choice.

In a classical seq2seq system, exactly the same decoding logic would be applied after conditioning the decoder on an encoder state or attention context.


In [ ]:
@torch.no_grad()
def prime_hidden(model: CharLSTM, prompt: str):
    token_ids = [stoi[ch] for ch in prompt]
    hidden = None
    if len(token_ids) > 1:
        prefix = torch.tensor(token_ids[:-1], dtype=torch.long, device=device).unsqueeze(0)
        _, hidden = model(prefix, hidden)
    last_token = torch.tensor([token_ids[-1]], dtype=torch.long, device=device)
    return token_ids, hidden, last_token


@torch.no_grad()
def greedy_decode(model: CharLSTM, prompt: str, max_new_tokens: int = 80):
    token_ids, hidden, last_token = prime_hidden(model, prompt)
    generated = token_ids[:]

    for _ in range(max_new_tokens):
        logits, hidden = model.step(last_token, hidden)
        next_token = int(torch.argmax(logits, dim=-1).item())
        generated.append(next_token)
        last_token = torch.tensor([next_token], dtype=torch.long, device=device)

    return decode(generated)


@torch.no_grad()
def beam_search_decode(model: CharLSTM, prompt: str, beam_width: int = 3, max_new_tokens: int = 80):
    prefix_ids, hidden, last_token = prime_hidden(model, prompt)
    beams = [(0.0, prefix_ids[:], hidden, last_token)]

    for _ in range(max_new_tokens):
        candidates = []
        for score, generated, hidden_state, token in beams:
            logits, next_hidden = model.step(token, hidden_state)
            log_probs = torch.log_softmax(logits, dim=-1).squeeze(0)
            top_scores, top_indices = torch.topk(log_probs, beam_width)

            for log_prob, token_id in zip(top_scores.tolist(), top_indices.tolist()):
                candidates.append(
                    (
                        score + log_prob,
                        generated + [token_id],
                        (next_hidden[0].clone(), next_hidden[1].clone()),
                        torch.tensor([token_id], dtype=torch.long, device=device),
                    )
                )

        candidates.sort(key=lambda item: item[0], reverse=True)
        beams = candidates[:beam_width]

    return [
        (score / max(len(tokens) - len(prefix_ids), 1), decode(tokens))
        for score, tokens, _, _ in beams
    ]


In [ ]:
prompt = "sequence "
print("greedy continuation:\n")
print(greedy_decode(model, prompt, max_new_tokens=90))

print("\nbeam-search continuations:\n")
for rank, (score, text) in enumerate(beam_search_decode(model, prompt, beam_width=3, max_new_tokens=80), start=1):
    print(f"beam {rank} | normalized log-prob {score:.4f}")
    print(text)
    print('-' * 60)


## Interpreting the decoding results

Even on a tiny corpus, the distinction between training and inference is visible:

- teacher forcing optimizes next-token prediction under the true prefix distribution;
- greedy decoding can get trapped by one locally attractive token choice;
- beam search keeps several live hypotheses, which is why it became standard in early sequence-to-sequence systems.

This is also the bridge to attention-based models: once sequence lengths grow, forcing a single recurrent state to store everything becomes a bottleneck, and decoding quality becomes sensitive to what information survives in that bottleneck.


## Recap and extension prompts

What this notebook established:

- an LSTM language model can be trained with teacher forcing using a shifted target sequence;
- the memory cell gives a more stable route for long-range information than a plain RNN hidden state;
- decoding is a separate algorithmic problem from training, and beam search is one classical answer.

Reasonable next steps:

1. Replace character tokens with words or subwords and compare the training dynamics.
2. Condition generation on an encoder state to turn the notebook into a small sequence-to-sequence example.
3. Compare greedy, beam, and temperature sampling on the same prompt.
